In [ ]:
# ══════════════════════════════════════════════════════════════════
# Colab setup — self-contained. No git clone, no FORK_URL, no sys.path.
# Before running: Runtime → Change runtime type → T4 GPU (free tier)
# ══════════════════════════════════════════════════════════════════
!pip install -q kailash polars plotly gdown python-dotenv nest-asyncio httpx kailash-ml kailash-kaizen

import nest_asyncio; nest_asyncio.apply()

import torch
if torch.cuda.is_available():
    print(f'✓ GPU available: {torch.cuda.get_device_name(0)}')
else:
    print('⚠ No GPU — training will be slow. Runtime → Change runtime type → T4 GPU')

print('✓ Setup complete. All helpers defined in the next cell.')


In [ ]:
# ══════════════════════════════════════════════════════════════════
# M6 Ollama bootstrap — local LLMs, no API keys
# On Colab: installs + starts Ollama and pulls this lesson's models.
# Locally: verifies the daemon is running and models are pulled.
# ══════════════════════════════════════════════════════════════════
_LESSON_MODELS = ['llama3.2:3b']
import sys, os, time, shutil, subprocess
import httpx

_OLLAMA_BASE_URL = os.environ.get('OLLAMA_BASE_URL', 'http://localhost:11434')
_IN_COLAB = 'google.colab' in sys.modules

def _daemon_up(timeout_s=1.0):
    try:
        r = httpx.get(f'{_OLLAMA_BASE_URL}/api/tags', timeout=timeout_s)
        r.raise_for_status()
        return r.json()
    except Exception:
        return None

if _IN_COLAB and shutil.which('ollama') is None:
    print('[ollama] installing binary …')
    subprocess.run(
        'curl -fsSL https://ollama.com/install.sh | sh',
        shell=True, check=True,
    )

if _IN_COLAB and _daemon_up() is None:
    print('[ollama] starting daemon …')
    subprocess.Popen(
        'nohup ollama serve > /tmp/ollama.log 2>&1 &',
        shell=True,
    )
    deadline = time.monotonic() + 30
    while time.monotonic() < deadline and _daemon_up() is None:
        time.sleep(0.5)

_tags = _daemon_up(timeout_s=2.0)
if _tags is None:
    raise RuntimeError(
        f'Ollama daemon not reachable at {_OLLAMA_BASE_URL}. '
        'On Colab: re-run this cell. Locally: run `ollama serve`.'
    )

_have = {m.get('name','').split(':',1)[0] for m in _tags.get('models', [])}
for _model in _LESSON_MODELS:
    _fam = _model.split(':', 1)[0]
    if _fam in _have:
        continue
    if _IN_COLAB:
        print(f'[ollama] pulling {_model} (one-time) …')
        subprocess.run(['ollama', 'pull', _model], check=True)
    else:
        raise RuntimeError(
            f'Model {_model!r} not pulled. Run: ollama pull {_model}'
        )

print(
    '✓ Ollama ready at', _OLLAMA_BASE_URL,
    '— models:', _LESSON_MODELS,
)



In [ ]:
# ══════════════════════════════════════════════════════════════════
# MLFP inlined helpers — DO NOT EDIT (collapse this cell!)
# Auto-generated by scripts/generate_selfcontained_notebook.py for mlfp06
# ══════════════════════════════════════════════════════════════════
from __future__ import annotations

# ── shared/kailash_helpers.py ──
"""Common Kailash SDK setup patterns for MLFP exercises."""


import logging
import os
from dataclasses import dataclass, field, replace
from pathlib import Path
from typing import Any, Literal

import numpy as np
import polars as pl
from dotenv import load_dotenv

logger = logging.getLogger(__name__)


def setup_environment() -> None:
    """Load .env and validate common configuration.

    Call this at the top of every exercise that needs API keys or DB connections.
    """
    # Find .env by walking up from the exercise file
    env_path = Path.cwd() / ".env"
    if not env_path.exists():
        # Try repo root
        for parent in Path.cwd().parents:
            candidate = parent / ".env"
            if candidate.exists():
                env_path = candidate
                break

    load_dotenv(env_path)


def get_connection_manager(db_url: str | None = None):
    """Create a ConnectionManager for kailash-ml engines.

    Args:
        db_url: Database URL. Defaults to SQLite at ./mlfp.db
    """
    from kailash.db import ConnectionManager

    url = db_url or os.environ.get("DATABASE_URL", "sqlite:///mlfp.db")
    return ConnectionManager(url)


def get_device() -> "torch.device":
    """Return the best available compute device as a ``torch.device``.

    Routes through ``kailash_ml.device()`` (the canonical kailash-ml 0.12+
    accelerator detector) so MPS / CUDA / ROCm / Intel XPU / CPU are all
    selected by the same logic the rest of the platform uses. The previous
    hand-rolled ``mps→cuda→cpu`` cascade is replaced because:

      * kailash-ml's detector knows about ROCm, Intel XPU, and fp16/bf16
        capability flags — the cascade in this helper did not.
      * Apple-Silicon students get the Metal Performance Shaders backend
        with mixed-precision (fp16) without any opt-in.
      * One detection point means lessons that print "Using device: …"
        agree with what kailash-ml's MLEngine() actually picks.
    """
    import kailash_ml as km
    import torch

    backend = km.device()  # BackendInfo (auto MPS on Mac, CUDA on Linux+NVIDIA, …)
    return torch.device(backend.device_string)


def get_llm_model() -> str:
    """Get the configured LLM model name from environment."""
    setup_environment()
    model = os.environ.get("DEFAULT_LLM_MODEL", os.environ.get("OPENAI_PROD_MODEL"))
    if not model:
        raise EnvironmentError(
            "No LLM model configured. Set DEFAULT_LLM_MODEL or OPENAI_PROD_MODEL in .env"
        )
    return model


# ════════════════════════════════════════════════════════════════════════
# SPLIT FIRST, THEN PREPROCESS
# ════════════════════════════════════════════════════════════════════════
# kailash-ml's ``PreprocessingPipeline.setup()`` fits its imputer, its
# categorical encoder (including TARGET encoding) and its scaler on EVERY
# row it is given, and only afterwards splits train/test. Handing it the
# whole dataset therefore lets test-set statistics leak into training.
#
# The helpers below hold the test rows out FIRST, fit the pipeline on the
# training rows only, and push the held-out rows through the fitted
# ``transform()``. ``setup()`` still splits internally, but that split now
# happens inside the training rows, so it cannot leak anything.

_SETUP_OPTIONS: frozenset[str] = frozenset(
    {
        "normalize",
        "normalize_method",
        "categorical_encoding",
        "imputation_strategy",
        "impute_n_neighbors",
        "remove_outliers",
        "outlier_threshold",
        "max_cardinality",
        "exclude_columns",
        "remove_multicollinearity",
        "multicollinearity_threshold",
        "pca",
        "pca_components",
        "transform_target",
        "fix_imbalance",
        "imbalance_method",
    }
)

# Arguments the helpers own themselves; passing them through would
# re-introduce the leak (``data`` = the full frame) or fight the split.
_OWNED_BY_HELPER: frozenset[str] = frozenset({"data", "target", "train_size", "seed"})


@dataclass(frozen=True)
class PreprocessResult:
    """Outcome of :func:`split_then_preprocess` / :func:`preprocess_train_test`.

    Mirrors the fields of kailash-ml's ``SetupResult`` that exercises use,
    plus the fitted pipeline and a record of what was done to which split.

    ``train_data`` / ``test_data``
        Both produced by the SAME pipeline, fitted on the training rows only.
    ``transformers``
        The fitted imputer / encoder / scaler state (e.g.
        ``transformers["ordinal_mappings"]``), learned from training rows.
    ``train_only_steps``
        Steps applied to the training split only (outlier removal,
        class-imbalance resampling). The test split is never filtered or
        resampled — it stays a faithful sample of the data.
    ``test_null_fills``
        Columns that were complete in the training rows but have gaps in
        the test rows (the fitted pipeline has no learned fill for them),
        mapped to the TRAINING statistic used to fill them.
    """

    train_data: pl.DataFrame
    test_data: pl.DataFrame
    target_column: str
    task_type: str
    numeric_columns: list[str]
    categorical_columns: list[str]
    transformers: dict[str, Any]
    pipeline: Any
    original_shape: tuple[int, int]
    train_raw_rows: int
    test_raw_rows: int
    stratified: bool
    train_only_steps: tuple[str, ...] = ()
    test_null_fills: dict[str, Any] = field(default_factory=dict)
    summary: str = ""


def _looks_like_classification(series: pl.Series) -> bool:
    """Same rule kailash-ml uses to call a target "classification"."""
    if series.dtype in (pl.Boolean, pl.Categorical, pl.Utf8, pl.String):
        return True
    return series.drop_nulls().n_unique() <= 20


def split_raw_train_test(
    df: pl.DataFrame,
    target: str,
    *,
    test_size: float | int = 0.2,
    seed: int = 42,
    stratify: bool | Literal["auto"] = "auto",
) -> tuple[pl.DataFrame, pl.DataFrame, bool]:
    """Hold out test rows from the RAW frame before anything is fitted.

    Rows are shuffled with ``seed``. ``test_size`` is a fraction in (0, 1)
    or an exact row count. With ``stratify="auto"`` a classification target
    (same rule as kailash-ml) is stratified so every class keeps its share
    in both splits; ``True`` / ``False`` force the choice.

    Returns ``(train_raw, test_raw, stratified)``; both splits keep the
    shuffled order.
    """
    if target not in df.columns:
        raise ValueError(f"Target column '{target}' not found in data.")
    n = df.height
    if n < 2:
        raise ValueError(f"Need at least 2 rows to split, got {n}.")
    if isinstance(test_size, np.integer):
        test_size = int(test_size)
    if isinstance(test_size, bool) or not isinstance(test_size, (int, float)):
        raise TypeError(
            f"test_size must be a fraction in (0, 1) or a row count, got {test_size!r}."
        )
    if isinstance(test_size, int):
        if not 1 <= test_size < n:
            raise ValueError(f"test_size as a row count must be in [1, {n - 1}], got {test_size}.")
        test_fraction = test_size / n
    else:
        if not 0.0 < test_size < 1.0:
            raise ValueError(f"test_size as a fraction must be in (0, 1), got {test_size}.")
        test_fraction = float(test_size)

    is_classification = _looks_like_classification(df[target])
    if stratify == "auto":
        stratified = is_classification
    elif stratify is True:
        if not is_classification:
            raise ValueError(
                f"stratify=True needs a classification target; '{target}' has "
                f"{df[target].drop_nulls().n_unique()} distinct values."
            )
        stratified = True
    elif stratify is False:
        stratified = False
    else:
        raise TypeError(f"stratify must be True, False or 'auto', got {stratify!r}.")

    perm = np.random.RandomState(seed).permutation(n)
    shuffled = df[perm.tolist()]

    if stratified:
        # Within each class (in shuffled order), the first round(n_c * f)
        # rows go to test — the class share is the same in both splits.
        is_test = pl.int_range(pl.len()).over(target) < (
            pl.len().over(target) * test_fraction
        ).round()
        flagged = shuffled.with_columns(is_test.alias("__split_is_test"))
        test_raw = flagged.filter(pl.col("__split_is_test")).drop("__split_is_test")
        train_raw = flagged.filter(~pl.col("__split_is_test")).drop("__split_is_test")
    else:
        n_test = test_size if isinstance(test_size, int) else int(round(n * test_fraction))
        train_raw = shuffled.head(n - n_test)
        test_raw = shuffled.tail(n_test)

    if train_raw.height == 0 or test_raw.height == 0:
        raise ValueError(
            f"Split produced an empty side (train={train_raw.height}, "
            f"test={test_raw.height}); adjust test_size."
        )
    return train_raw, test_raw, stratified


def _fill_unfitted_test_nulls(
    test_raw: pl.DataFrame,
    train_raw: pl.DataFrame,
    fitted: Any,
    strategy: str,
) -> tuple[pl.DataFrame, dict[str, Any]]:
    """Fill test gaps the fitted pipeline has no learned value for.

    kailash-ml only stores a fill value for columns that had gaps in the
    rows it was fitted on. A column that is complete in the training rows
    but has gaps in the test rows would otherwise reach the model as null.
    The fill is the TRAINING statistic for the same strategy (never a test
    statistic), and every such column is logged and returned.
    """
    if strategy == "drop":
        return test_raw, {}  # transform() drops incomplete rows itself
    learned = set(fitted.transformers.get("imputer_stats", {}))
    if strategy in ("knn", "iterative"):
        learned |= set(fitted.transformers.get("sklearn_imputer_cols", []))

    fills: dict[str, Any] = {}
    for col in fitted.numeric_columns + fitted.categorical_columns:
        if col in learned or col not in test_raw.columns:
            continue
        if test_raw[col].null_count() == 0:
            continue
        train_col = train_raw[col].drop_nulls()
        if col in fitted.categorical_columns or strategy == "mode":
            modes = train_col.mode().sort()
            if len(modes) == 0:
                raise ValueError(f"Cannot fill '{col}': no observed training values.")
            value = modes[0]
        elif strategy == "median":
            value = train_col.median()
        else:  # mean (and the numeric fallback for any other strategy)
            value = train_col.mean()
        if value is None:
            raise ValueError(f"Cannot fill '{col}': no observed training values.")
        fills[col] = value
    if fills:
        logger.warning(
            "Test rows have gaps in columns that were complete in training; "
            "filled with TRAINING statistics: %s",
            fills,
        )
        test_raw = test_raw.with_columns(
            [pl.col(c).fill_null(v) for c, v in fills.items()]
        )
    return test_raw, fills


def preprocess_train_test(
    train_raw: pl.DataFrame,
    test_raw: pl.DataFrame,
    target: str,
    *,
    seed: int = 42,
    **setup_kwargs: Any,
) -> PreprocessResult:
    """Fit ``PreprocessingPipeline`` on ``train_raw`` only; transform both.

    ``setup_kwargs`` are ``PreprocessingPipeline.setup()`` options
    (``normalize``, ``categorical_encoding``, ``imputation_strategy`` …).
    Steps that change the ROWS of a split are applied to training rows only:

    * ``remove_outliers`` — outlier rows are dropped from training only.
    * ``fix_imbalance`` — SMOTE / ADASYN (or the class-weight flag) is
      applied to the whole transformed training split, never to test.

    Use this directly inside cross-validation folds; use
    :func:`split_then_preprocess` for a single train/test split.
    """
    from kailash_ml import PreprocessingPipeline

    owned = sorted(_OWNED_BY_HELPER & set(setup_kwargs))
    if owned:
        raise TypeError(
            f"{owned} are set by the split-first helper itself and cannot be "
            "passed through to setup()."
        )
    unknown = sorted(set(setup_kwargs) - _SETUP_OPTIONS)
    if unknown:
        raise TypeError(f"Unknown PreprocessingPipeline.setup() options: {unknown}.")
    for name, frame in (("train_raw", train_raw), ("test_raw", test_raw)):
        if target not in frame.columns:
            raise ValueError(f"Target column '{target}' not found in {name}.")
    if train_raw.height == 0 or test_raw.height == 0:
        raise ValueError("train_raw and test_raw must both have rows.")
    if set(train_raw.columns) != set(test_raw.columns):
        raise ValueError("train_raw and test_raw must have the same columns.")

    fix_imbalance = bool(setup_kwargs.pop("fix_imbalance", False))
    imbalance_method = setup_kwargs.get("imbalance_method", "smote")
    remove_outliers = bool(setup_kwargs.get("remove_outliers", False))

    pipeline = PreprocessingPipeline()
    # The pipeline only ever sees training rows. Its internal split stays
    # inside them; resampling is applied below to the full training split.
    fitted = pipeline.setup(
        data=train_raw, target=target, seed=seed, fix_imbalance=False, **setup_kwargs
    )

    train_only: list[str] = []
    if remove_outliers:
        # setup() removed the outlier rows (training rows only); its two
        # internal pieces together are every surviving training row.
        train_data = pl.concat([fitted.train_data, fitted.test_data], how="vertical")
        train_only.append(
            f"remove_outliers: {train_raw.height - train_data.height} training rows dropped"
        )
    else:
        train_data = pipeline.transform(train_raw)

    strategy = setup_kwargs.get("imputation_strategy", "mean")
    test_filled, test_fills = _fill_unfitted_test_nulls(test_raw, train_raw, fitted, strategy)
    test_data = pipeline.transform(test_filled)

    if fix_imbalance and fitted.task_type == "classification":
        resample = getattr(pipeline, "_apply_imbalance_correction", None)
        if resample is None:
            raise RuntimeError(
                "This kailash-ml version has no imbalance-correction step on "
                "PreprocessingPipeline; cannot apply fix_imbalance to the training split."
            )
        before = train_data.height
        train_data = resample(train_data, target, imbalance_method, seed)
        train_only.append(
            f"fix_imbalance ({imbalance_method}): training rows {before} -> {train_data.height}"
        )

    summary = "\n".join(
        [
            f"Task type: {fitted.task_type}",
            f"Split first: {train_raw.height} train rows / {test_raw.height} test rows "
            "(pipeline fitted on the train rows only)",
            f"Numeric features: {len(fitted.numeric_columns)}",
            f"Categorical features: {len(fitted.categorical_columns)}",
            f"Encoding: {setup_kwargs.get('categorical_encoding', 'onehot')}",
            f"Normalization: {'yes' if setup_kwargs.get('normalize', True) else 'no'}",
            f"Imputation: {strategy}",
            f"Train-only steps: {'; '.join(train_only) if train_only else 'none'}",
            f"Transformed train: {train_data.height} rows x {train_data.width} cols",
            f"Transformed test: {test_data.height} rows x {test_data.width} cols",
        ]
    )

    return PreprocessResult(
        train_data=train_data,
        test_data=test_data,
        target_column=target,
        task_type=fitted.task_type,
        numeric_columns=list(fitted.numeric_columns),
        categorical_columns=list(fitted.categorical_columns),
        transformers=dict(fitted.transformers),
        pipeline=pipeline,
        original_shape=(train_raw.height + test_raw.height, train_raw.width),
        train_raw_rows=train_raw.height,
        test_raw_rows=test_raw.height,
        stratified=False,
        train_only_steps=tuple(train_only),
        test_null_fills=test_fills,
        summary=summary,
    )


def split_then_preprocess(
    df: pl.DataFrame,
    target: str,
    *,
    test_size: float | int = 0.2,
    seed: int = 42,
    stratify: bool | Literal["auto"] = "auto",
    **setup_kwargs: Any,
) -> PreprocessResult:
    """Leak-free replacement for ``PreprocessingPipeline().setup(df, ...)``.

    1. Split the RAW frame first (seeded shuffle; stratified on the target
       for classification when ``stratify="auto"``).
    2. Fit ``PreprocessingPipeline`` on the training rows only.
    3. ``transform()`` the test rows with the fitted pipeline.

    ``setup_kwargs`` are passed to ``PreprocessingPipeline.setup()``; see
    :func:`preprocess_train_test` for how row-changing options are handled.
    """
    train_raw, test_raw, stratified = split_raw_train_test(
        df, target, test_size=test_size, seed=seed, stratify=stratify
    )
    result = preprocess_train_test(train_raw, test_raw, target, seed=seed, **setup_kwargs)
    split_line = "stratified on the target" if stratified else "random"
    return replace(
        result,
        stratified=stratified,
        summary=result.summary.replace("Split first:", f"Split first ({split_line}):", 1),
    )


# ── shared/mlfp06/_ollama_bootstrap.py ──
"""Ollama bootstrap for MLFP06 — local LLMs, no API keys, real calls only.

Why this module exists
----------------------
M6 trains, prompts, evaluates, and deploys real LLMs. Routing every call
through OpenAI:

  * forces every learner to hold an API key
  * burns budget every time a notebook reruns a cell
  * makes "did the LLM actually fire?" invisible (a stubbed `cost=$0`
    looks identical to a successful call)

This module switches every M6 LLM call to a locally-running Ollama daemon
that the student installs once and re-uses across every lesson.

Contract
--------
* :func:`make_delegate` — the only sanctioned way to construct a Kaizen
  ``Delegate`` for M6. Wraps the Ollama adapter with sensible defaults and
  forces ``budget_usd=None`` (Kaizen mis-prices Ollama as $3/$15-per-Mtok;
  budget tracking against a free local provider is meaningless).
* :func:`make_embedder` — the only sanctioned way to construct an embedding
  adapter for M6 RAG (lesson 6.4).
* :func:`preflight_ollama` — pre-flight reachability + model-presence check.
  Raises :class:`OllamaUnreachableError` with an actionable message. Call
  this at the top of every notebook / script before constructing a
  Delegate.
* :data:`LESSON_MODELS` — manifest of which models each lesson needs.
  Consumed by ``scripts/generate_selfcontained_notebook.py`` to inject the
  Colab bootstrap cell.

Per ``rules/zero-tolerance.md`` Rule 2: there is no silent stub fallback
in this module. If Ollama is not reachable, we raise. The previous
``run_delegate(...) -> ("unknown", 0.0, latency)`` fallback is BLOCKED.

Per ``rules/framework-first.md``: every LLM call goes through Kaizen
``Delegate``. Raw ``httpx`` calls to ``/api/generate`` are BLOCKED for
generation paths; ``httpx`` is used here only for the daemon-reachability
preflight.
"""

import logging
import os
import shutil
import subprocess
import sys
import time
from typing import Any

import httpx
from dotenv import load_dotenv

logger = logging.getLogger(__name__)

# ════════════════════════════════════════════════════════════════════════
# Configuration — env-driven per rules/env-models.md
# ════════════════════════════════════════════════════════════════════════

load_dotenv()


def _env(key: str, default: str) -> str:
    val = os.environ.get(key)
    return val if val else default


OLLAMA_BASE_URL: str = _env("OLLAMA_BASE_URL", "http://localhost:11434")
"""Where the Ollama daemon listens. Local default; Colab uses the same."""

DEFAULT_CHAT_MODEL: str = _env("OLLAMA_CHAT_MODEL", "llama3.2:3b")
"""Tool-capable instruct model. Allowlisted by Kaizen for function calling."""

DEFAULT_EMBED_MODEL: str = _env("OLLAMA_EMBED_MODEL", "nomic-embed-text")
"""Compact embedding model — ~270MB, 768-dim. Used by M6 RAG (lesson 6.4)."""

DEFAULT_FT_BASE_MODEL: str = _env("OLLAMA_FT_BASE_MODEL", "qwen2.5:0.5b")
"""Tiny base model for fine-tuning demos (lessons 6.2, 6.3). Fits T4."""


# ════════════════════════════════════════════════════════════════════════
# Lesson → models manifest (used by generator + preflight)
# ════════════════════════════════════════════════════════════════════════


LESSON_MODELS: dict[str, list[str]] = {
    # 6.1 Prompting — chat only
    "ex_1": [DEFAULT_CHAT_MODEL],
    # 6.2 Fine-tuning — base for LoRA + chat for evaluation
    "ex_2": [DEFAULT_FT_BASE_MODEL, DEFAULT_CHAT_MODEL],
    # 6.3 Alignment (DPO/GRPO) — base + chat for judge
    "ex_3": [DEFAULT_FT_BASE_MODEL, DEFAULT_CHAT_MODEL],
    # 6.4 RAG — chat + embeddings
    "ex_4": [DEFAULT_CHAT_MODEL, DEFAULT_EMBED_MODEL],
    # 6.5 Agents (ReAct) — tool-capable chat
    "ex_5": [DEFAULT_CHAT_MODEL],
    # 6.6 Multi-agent + MCP — tool-capable chat
    "ex_6": [DEFAULT_CHAT_MODEL],
    # 6.7 PACT governance — chat for governed Delegate
    "ex_7": [DEFAULT_CHAT_MODEL],
    # 6.8 Nexus deployment — chat behind the API
    "ex_8": [DEFAULT_CHAT_MODEL],
}


# ════════════════════════════════════════════════════════════════════════
# Errors
# ════════════════════════════════════════════════════════════════════════


class OllamaUnreachableError(RuntimeError):
    """Daemon not running, daemon unreachable, or required model not pulled.

    The error message MUST be actionable: it tells the user the exact
    command to run to fix the situation. We never wrap a low-level
    ``httpx.ConnectError`` and re-raise without translating it.
    """


# ════════════════════════════════════════════════════════════════════════
# Preflight — reachability + model presence
# ════════════════════════════════════════════════════════════════════════


def _running_in_colab() -> bool:
    return "google.colab" in sys.modules


def _list_pulled_families(*, timeout_s: float = 2.0) -> set[str]:
    """Hit /api/tags and return the set of pulled model families (no tag).

    Returns ``{"llama3.2", "qwen2.5"}`` for a daemon with
    ``llama3.2:3b`` and ``qwen2.5:0.5b`` pulled. The model family is
    everything before the first colon — we match against family rather
    than the full ``family:tag`` because pulling ``llama3.2:3b-instruct``
    or ``llama3.2:3b-instruct-q4_K_M`` should still satisfy a request
    for the ``llama3.2`` family.
    """
    url = f"{OLLAMA_BASE_URL.rstrip('/')}/api/tags"
    try:
        r = httpx.get(url, timeout=timeout_s)
        r.raise_for_status()
    except (httpx.HTTPError, httpx.ConnectError) as exc:
        raise OllamaUnreachableError(
            f"ollama daemon not reachable at {OLLAMA_BASE_URL}. "
            f"Fix: open a terminal and run `ollama serve`. "
            f"On macOS the desktop app starts the daemon automatically. "
            f"Underlying: {type(exc).__name__}: {exc}"
        ) from exc
    payload = r.json() or {}
    families: set[str] = set()
    for entry in payload.get("models", []):
        name = entry.get("name") or entry.get("model") or ""
        if not name:
            continue
        families.add(name.split(":", 1)[0])
    return families


def preflight_ollama(
    *,
    required_models: list[str] | None = None,
    auto_pull: bool | None = None,
    timeout_s: float = 2.0,
) -> None:
    """Verify the daemon is reachable and the required models are pulled.

    Args:
        required_models: Models that must be present, e.g.
            ``["llama3.2:3b", "nomic-embed-text"]``. Family-matched (the
            ``:tag`` portion is informational; any pulled tag for the
            same family satisfies the requirement).
        auto_pull: If ``True``, attempt ``ollama pull <model>`` for any
            missing model via subprocess. Defaults to ``True`` on Colab
            (where the bootstrap cell has just installed the binary)
            and ``False`` locally (where pulling is the student's job
            and surprise downloads are unwelcome).
        timeout_s: HTTP timeout for the reachability check. The default
            is intentionally short — if the daemon is alive it answers
            within 100ms.

    Raises:
        OllamaUnreachableError: Daemon unreachable, or a required model
            is missing after pull (or auto_pull was disabled).
    """
    if auto_pull is None:
        auto_pull = _running_in_colab()

    have = _list_pulled_families(timeout_s=timeout_s)
    needed = required_models or []
    missing = [m for m in needed if m.split(":", 1)[0] not in have]

    if missing and auto_pull:
        for model in missing:
            _pull_model(model)
        # Re-check
        have = _list_pulled_families(timeout_s=timeout_s)
        missing = [m for m in needed if m.split(":", 1)[0] not in have]

    if missing:
        cmds = "\n  ".join(f"ollama pull {m}" for m in missing)
        raise OllamaUnreachableError(
            f"Required model(s) not pulled: {missing}.\n"
            f"Fix — run in a terminal:\n  {cmds}"
        )

    logger.info(
        "ollama.preflight.ok",
        extra={
            "base_url": OLLAMA_BASE_URL,
            "models_present": sorted(have),
            "required": needed,
        },
    )


def _pull_model(model: str) -> None:
    """Run ``ollama pull <model>`` as a subprocess. Streams to stdout."""
    if shutil.which("ollama") is None:
        raise OllamaUnreachableError(
            f"`ollama` binary not on PATH — cannot pull {model}. "
            f"Install: https://ollama.com/download"
        )
    print(f"[ollama] pulling {model} (one-time download) …", flush=True)
    t0 = time.monotonic()
    proc = subprocess.run(
        ["ollama", "pull", model],
        capture_output=False,
        text=True,
    )
    elapsed = time.monotonic() - t0
    if proc.returncode != 0:
        raise OllamaUnreachableError(
            f"`ollama pull {model}` failed with exit {proc.returncode} "
            f"after {elapsed:.1f}s. Check network and disk space."
        )
    print(f"[ollama] {model} pulled in {elapsed:.1f}s", flush=True)


# ════════════════════════════════════════════════════════════════════════
# Delegate factory — the only sanctioned constructor for M6
# ════════════════════════════════════════════════════════════════════════


def make_delegate(
    *,
    model: str | None = None,
    tools: Any = None,
    system_prompt: str | None = None,
    temperature: float = 0.4,
    max_tokens: int = 4096,
    **kwargs: Any,
) -> Any:
    """Construct a Kaizen ``Delegate`` backed by the local Ollama daemon.

    This is the ONLY way M6 helpers and exercises construct a Delegate.
    Direct ``Delegate(...)`` construction in M6 code is BLOCKED — the
    cost-mispricing trap (``budget_usd`` against a free provider) and
    the silent OpenAI fallback that comes from omitting ``adapter=`` are
    the reasons.

    Args:
        model: Ollama model tag (e.g. ``"llama3.2:3b"``). Defaults to
            :data:`DEFAULT_CHAT_MODEL`.
        tools: Tool registry or list — passed through to Delegate.
            Auto-disables streaming on the Ollama adapter.
        system_prompt: Optional system prompt.
        temperature: Sampling temperature. Default 0.4.
        max_tokens: Hard cap on output length. Default 4096.
        **kwargs: Forwarded to ``Delegate(...)``. ``budget_usd`` is
            forced to ``None`` and ``api_key`` / ``base_url`` are
            BLOCKED (the adapter owns those).

    Returns:
        A configured Kaizen ``Delegate`` instance.
    """
    for blocked in ("api_key", "base_url", "budget_usd", "adapter"):
        if blocked in kwargs:
            raise ValueError(
                f"make_delegate(): {blocked!r} is owned by the bootstrap "
                f"and cannot be overridden by callers."
            )

    # Lazy imports — keep this module importable in environments that have
    # not yet installed kaizen-agents (e.g. early bootstrap cells).
    from kaizen_agents import Delegate
    from kaizen_agents.delegate.adapters.registry import get_adapter

    resolved = model or DEFAULT_CHAT_MODEL
    adapter = get_adapter(
        "ollama",
        model=resolved,
        base_url=OLLAMA_BASE_URL,
        temperature=temperature,
        max_tokens=max_tokens,
    )
    return Delegate(
        model=resolved,
        adapter=adapter,
        tools=tools,
        system_prompt=system_prompt,
        budget_usd=None,
        **kwargs,
    )


# ════════════════════════════════════════════════════════════════════════
# Embedding factory — the only sanctioned constructor for M6 RAG
# ════════════════════════════════════════════════════════════════════════


def make_embedder(*, model: str | None = None) -> Any:
    """Construct a Kaizen embedding adapter backed by Ollama.

    Args:
        model: Ollama embedding model tag. Defaults to
            :data:`DEFAULT_EMBED_MODEL`.

    Returns:
        An adapter with ``async def embed(texts: list[str]) -> list[list[float]]``.
    """
    from kaizen_agents.delegate.adapters.registry import get_embedding_adapter

    resolved = model or DEFAULT_EMBED_MODEL
    return get_embedding_adapter(
        "ollama",
        model=resolved,
        base_url=OLLAMA_BASE_URL,
    )


# ════════════════════════════════════════════════════════════════════════
# Streaming helpers — same call shape as the old `run_delegate`
# ════════════════════════════════════════════════════════════════════════


async def run_delegate_text(
    delegate: Any, prompt: str
) -> tuple[str, dict[str, int], float]:
    """Run a Delegate to completion, return ``(text, usage, latency_s)``.

    ``usage`` is a dict with ``prompt_tokens``, ``completion_tokens``,
    ``total_tokens`` (zero for events that don't carry usage). ``cost``
    is intentionally absent — Ollama is free, and Kaizen's ``consumed_usd``
    is mis-priced for local models.

    Event handling: the Kaizen ``TextDelta`` events expose the streamed
    chunk on ``.text`` (not ``.delta_text`` — that name is reserved for a
    different StreamEvent shape). The terminal ``TurnComplete`` event
    carries the final ``.text`` (entire response, not a delta) plus the
    ``.usage`` dict. We accumulate from ``TextDelta.text`` for streaming
    UX and fall back to ``TurnComplete.text`` if no deltas arrived.
    """
    t0 = time.perf_counter()
    response = ""
    usage: dict[str, int] = {
        "prompt_tokens": 0,
        "completion_tokens": 0,
        "total_tokens": 0,
    }
    final_text: str | None = None
    async for event in delegate.run(prompt):
        ev_type = getattr(event, "event_type", None)
        text = getattr(event, "text", None)
        if ev_type == "text_delta" and text:
            response += text
        elif ev_type == "turn_complete":
            final_text = text or final_text
            ev_usage = getattr(event, "usage", None) or {}
            for k in usage:
                if k in ev_usage:
                    usage[k] = int(ev_usage[k])
    if not response and final_text:
        response = final_text
    elapsed = time.perf_counter() - t0
    return response, usage, elapsed


# ════════════════════════════════════════════════════════════════════════
# Colab installer — used by the generator's Cell 0
# ════════════════════════════════════════════════════════════════════════


def colab_install_and_start_ollama(*, log_path: str = "/tmp/ollama.log") -> None:
    """Install the Ollama binary and start the daemon (Colab only).

    Idempotent: skips install if the binary is already on PATH and
    skips daemon start if a process is already listening on
    :data:`OLLAMA_BASE_URL`. Raises :class:`OllamaUnreachableError`
    if either step fails.

    Not for local use — locally, students run ``ollama serve`` themselves.
    """
    if not _running_in_colab():
        raise RuntimeError(
            "colab_install_and_start_ollama() is Colab-only. "
            "On a local machine, run `ollama serve` in a terminal."
        )

    # Step 1 — install if missing
    if shutil.which("ollama") is None:
        print("[colab] installing ollama …", flush=True)
        proc = subprocess.run(
            "curl -fsSL https://ollama.com/install.sh | sh",
            shell=True,
            check=False,
            capture_output=True,
            text=True,
        )
        if proc.returncode != 0:
            raise OllamaUnreachableError(
                f"ollama install failed (exit {proc.returncode}):\n"
                f"stdout: {proc.stdout[-500:]}\nstderr: {proc.stderr[-500:]}"
            )

    # Step 2 — start daemon if not already listening
    try:
        httpx.get(f"{OLLAMA_BASE_URL}/api/tags", timeout=1.0).raise_for_status()
        print("[colab] ollama daemon already running", flush=True)
        return
    except (httpx.HTTPError, httpx.ConnectError):
        pass

    print(f"[colab] starting ollama daemon (logs → {log_path}) …", flush=True)
    subprocess.Popen(
        f"nohup ollama serve > {log_path} 2>&1 &",
        shell=True,
        stdout=subprocess.DEVNULL,
        stderr=subprocess.DEVNULL,
    )

    # Step 3 — wait up to 30s for the daemon to answer /api/tags
    deadline = time.monotonic() + 30.0
    last_err: Exception | None = None
    while time.monotonic() < deadline:
        try:
            httpx.get(f"{OLLAMA_BASE_URL}/api/tags", timeout=1.0).raise_for_status()
            print("[colab] ollama daemon ready", flush=True)
            return
        except (httpx.HTTPError, httpx.ConnectError) as exc:
            last_err = exc
            time.sleep(0.5)

    raise OllamaUnreachableError(
        f"ollama daemon failed to come up within 30s. "
        f"Last error: {type(last_err).__name__}: {last_err}. "
        f"Inspect {log_path} for details."
    )


# ── shared/mlfp06/ex_6.py ──
"""
Shared infrastructure for Exercise 6 — Multi-Agent Orchestration and MCP.

Contains: SQuAD 2.0 corpus loading, specialist Signature definitions,
specialist BaseAgent classes, synthesis agent, output directory setup, and
run_checked() — the guard that turns a failed LLM call into a loud error.
Technique-specific orchestration logic lives in the per-technique files.
"""

from dataclasses import dataclass, field
from pathlib import Path

import polars as pl

from kaizen import InputField, OutputField, Signature
from kaizen.core.base_agent import BaseAgent


# ════════════════════════════════════════════════════════════════════════
# ENVIRONMENT SETUP
# ════════════════════════════════════════════════════════════════════════

setup_environment()


MODEL = DEFAULT_CHAT_MODEL
# Every M6 agent runs on the local Ollama daemon; the model comes from
# OLLAMA_CHAT_MODEL and the address from OLLAMA_BASE_URL (both in .env).
LLM_PROVIDER_DEFAULT = "ollama"
LLM_BASE_URL_DEFAULT = OLLAMA_BASE_URL

OLLAMA_HINT = (
    "Is the local Ollama daemon running? Start it with `ollama serve` and "
    f"pull the chat model with `ollama pull {MODEL}`."
)

# Output directory for all visualisation/trace artifacts
OUTPUT_DIR = Path("outputs") / "ex6_multi_agent"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)


# ════════════════════════════════════════════════════════════════════════
# DATA LOADING — SQuAD 2.0 Multi-Domain Corpus
# ════════════════════════════════════════════════════════════════════════

CACHE_DIR = Path("data") / "mlfp06" / "squad"
CACHE_DIR.mkdir(parents=True, exist_ok=True)
CACHE_FILE = CACHE_DIR / "squad_v2_300.parquet"


def load_squad_corpus(n_rows: int = 300) -> pl.DataFrame:
    """Load (or download + cache) SQuAD 2.0 validation split.

    SQuAD 2.0 is a multi-domain reading-comprehension benchmark with
    100K+ questions across hundreds of Wikipedia titles. We take a
    shuffled slice of 300 rows so exercises run in bounded time while
    still exercising the "multi-domain" property.

    Returns a polars DataFrame with columns: title, text, question, answer.
    """
    if CACHE_FILE.exists():
        print(f"Loading cached SQuAD 2.0 from {CACHE_FILE}")
        return pl.read_parquet(CACHE_FILE)

    print("Downloading rajpurkar/squad_v2 from HuggingFace...")
    from datasets import load_dataset

    ds = load_dataset("rajpurkar/squad_v2", split="validation")
    ds = ds.shuffle(seed=42).select(range(min(n_rows, len(ds))))
    rows = []
    for row in ds:
        answers = row["answers"]["text"]
        rows.append(
            {
                "title": row["title"],
                "text": row["context"],
                "question": row["question"],
                "answer": answers[0] if answers else "",
            }
        )
    passages = pl.DataFrame(rows)
    passages.write_parquet(CACHE_FILE)
    print(f"Cached {passages.height} SQuAD 2.0 rows to {CACHE_FILE}")
    return passages


# ════════════════════════════════════════════════════════════════════════
# SPECIALIST SIGNATURES — Domain-Specific Input/Output Contracts
# ════════════════════════════════════════════════════════════════════════


class FactualAnalysisSignature(Signature):
    """Extract factual claims and verify them against the passage."""

    document: str = InputField(description="Source passage text")
    question: str = InputField(description="Question to analyse")
    factual_claims: list[str] = OutputField(
        description="Key factual claims in the passage"
    )
    evidence_quality: str = OutputField(
        description="Quality of evidence: strong/moderate/weak"
    )
    answer_supported: bool = OutputField(
        description="Whether the passage supports an answer"
    )


class SemanticAnalysisSignature(Signature):
    """Analyse meaning, context, and implications beyond literal text."""

    document: str = InputField(description="Source passage text")
    question: str = InputField(description="Question to analyse")
    main_themes: list[str] = OutputField(description="Central themes in the passage")
    implicit_info: list[str] = OutputField(
        description="Implied but not stated information"
    )
    contextual_relevance: str = OutputField(
        description="How relevant context is to the question"
    )


class StructuralAnalysisSignature(Signature):
    """Analyse the structure, organisation, and argumentation pattern."""

    document: str = InputField(description="Source passage text")
    question: str = InputField(description="Question to analyse")
    structure_type: str = OutputField(
        description="Text structure: narrative/expository/argumentative"
    )
    key_entities: list[str] = OutputField(description="Named entities mentioned")
    relationships: list[str] = OutputField(description="Relationships between entities")


class SynthesisSignature(Signature):
    """Synthesise multiple specialist analyses into a unified answer."""

    document: str = InputField(description="Original passage")
    question: str = InputField(description="Original question")
    factual_analysis: str = InputField(description="Factual specialist output")
    semantic_analysis: str = InputField(description="Semantic specialist output")
    structural_analysis: str = InputField(description="Structural specialist output")
    unified_answer: str = OutputField(
        description="Comprehensive answer drawing on all analyses"
    )
    confidence: float = OutputField(description="Answer confidence 0-1")
    reasoning_chain: list[str] = OutputField(description="Step-by-step reasoning used")


class InterpretationSignature(Signature):
    """Interpret factual claims in context (sequential pipeline stage 2)."""

    factual_claims: str = InputField(description="Raw factual claims from prior stage")
    document: str = InputField(description="Original passage for context")
    question: str = InputField(description="Question being answered")
    interpreted_facts: list[str] = OutputField(
        description="Facts with contextual interpretation"
    )
    relevance_ranking: list[str] = OutputField(
        description="Facts ranked by relevance to question"
    )


# ════════════════════════════════════════════════════════════════════════
# SPECIALIST AGENT CLASSES
# ════════════════════════════════════════════════════════════════════════
#
# Canonical kaizen 2.7.3 pattern: dataclass config + instance signature
# in super().__init__.  Class-level `signature = XxxSig` is a silent
# bug — BaseAgent ignores class-level attrs and falls back to
# DefaultSignature, so every "specialist" was producing generic output
# before this migration.  See workspaces/mlfp06-migration/api-cheatsheet.md
# for the background.
#
# `description` remains a class-level attribute because it's read by
# Pipeline.router() and by the exercise's audit-trail logging — it's
# agent metadata, not part of the LLM-wiring contract.


# kaizen 2.28 wiring contract for BaseAgent specialists:
#
#   1. `use_async_llm=True` — the exercises invoke `await agent.run_async(...)`.
#      Without this flag BaseAgent never constructs the async LLM client and
#      run_async() raises ValueError("Agent not configured for async mode").
#
#   2. `response_format={"type": "json_object"}` + `structured_output_mode=
#      "explicit"` — in 2.28 the default mode is "explicit", which (unlike the
#      old "auto") does NOT inject JSON-format instructions unless a
#      response_format is declared. Without it the model returns prose, the
#      strategy reports error="JSON_PARSE_FAILED", and the typed Signature
#      fields (factual_claims, …) are never unpacked into the result dict —
#      the exercises then KeyError on `result['factual_claims']`.
#
# from_domain_config() forwards all three fields automatically, so declaring
# them on each domain config is the canonical, single-source fix for every
# ex_6 solution and student file.
def _json_object_format() -> dict[str, str]:
    return {"type": "json_object"}


@dataclass
class FactualConfig:
    llm_provider: str = LLM_PROVIDER_DEFAULT
    model: str = MODEL
    base_url: str = LLM_BASE_URL_DEFAULT
    temperature: float = 0.2
    budget_limit_usd: float = 1.0
    use_async_llm: bool = True
    response_format: dict = field(default_factory=_json_object_format)
    structured_output_mode: str = "explicit"


@dataclass
class SemanticConfig:
    llm_provider: str = LLM_PROVIDER_DEFAULT
    model: str = MODEL
    base_url: str = LLM_BASE_URL_DEFAULT
    temperature: float = 0.2
    budget_limit_usd: float = 1.0
    use_async_llm: bool = True
    response_format: dict = field(default_factory=_json_object_format)
    structured_output_mode: str = "explicit"


@dataclass
class StructuralConfig:
    llm_provider: str = LLM_PROVIDER_DEFAULT
    model: str = MODEL
    base_url: str = LLM_BASE_URL_DEFAULT
    temperature: float = 0.2
    budget_limit_usd: float = 1.0
    use_async_llm: bool = True
    response_format: dict = field(default_factory=_json_object_format)
    structured_output_mode: str = "explicit"


@dataclass
class SynthesisConfig:
    llm_provider: str = LLM_PROVIDER_DEFAULT
    model: str = MODEL
    base_url: str = LLM_BASE_URL_DEFAULT
    temperature: float = 0.2
    # Supervisor gets a larger budget — it reasons over all specialist outputs.
    budget_limit_usd: float = 2.0
    use_async_llm: bool = True
    response_format: dict = field(default_factory=_json_object_format)
    structured_output_mode: str = "explicit"


@dataclass
class InterpretationConfig:
    llm_provider: str = LLM_PROVIDER_DEFAULT
    model: str = MODEL
    base_url: str = LLM_BASE_URL_DEFAULT
    temperature: float = 0.2
    budget_limit_usd: float = 1.0
    use_async_llm: bool = True
    response_format: dict = field(default_factory=_json_object_format)
    structured_output_mode: str = "explicit"


class FactualAgent(BaseAgent):
    description = "Specialist in factual analysis: claims, evidence, verification"

    def __init__(self, config: FactualConfig | None = None):
        super().__init__(
            config=config or FactualConfig(),
            signature=FactualAnalysisSignature(),
        )


class SemanticAgent(BaseAgent):
    description = "Specialist in semantic analysis: themes, implications, context"

    def __init__(self, config: SemanticConfig | None = None):
        super().__init__(
            config=config or SemanticConfig(),
            signature=SemanticAnalysisSignature(),
        )


class StructuralAgent(BaseAgent):
    description = (
        "Specialist in structural analysis: entities, relationships, organisation"
    )

    def __init__(self, config: StructuralConfig | None = None):
        super().__init__(
            config=config or StructuralConfig(),
            signature=StructuralAnalysisSignature(),
        )


class SynthesisAgent(BaseAgent):
    description = (
        "Supervisor that synthesises specialist analyses into unified decisions"
    )

    def __init__(self, config: SynthesisConfig | None = None):
        super().__init__(
            config=config or SynthesisConfig(),
            signature=SynthesisSignature(),
        )


class InterpretationAgent(BaseAgent):
    description = "Stage-2 interpreter: contextualises raw factual claims for synthesis"

    def __init__(self, config: InterpretationConfig | None = None):
        super().__init__(
            config=config or InterpretationConfig(),
            signature=InterpretationSignature(),
        )


async def run_checked(agent: BaseAgent, **inputs: object) -> dict:
    """Await ``agent.run_async(**inputs)`` and raise if the LLM call failed.

    ``BaseAgent.run_async`` reports a provider failure (daemon down, model
    missing, unparseable reply) as ``{"error": ..., "success": False}``
    instead of raising. Without this guard the failure only surfaces later
    as a confusing ``KeyError`` on a missing output field.

    Args:
        agent: Any specialist / synthesis BaseAgent.
        **inputs: The Signature's input fields.

    Returns:
        The agent's typed result dict.
    """
    result = await agent.run_async(**inputs)
    if not isinstance(result, dict) or "error" in result or result.get("success") is False:
        detail = result.get("error") if isinstance(result, dict) else repr(result)
        raise RuntimeError(
            f"{agent.__class__.__name__} LLM call failed: {detail}. {OLLAMA_HINT}"
        )
    return result


def build_specialists() -> tuple[FactualAgent, SemanticAgent, StructuralAgent]:
    """Return fresh instances of the three analysis specialists.

    Uses the default Config for each specialist.  Callers that need to
    tweak model, temperature, or budget can instantiate directly:

        factual = FactualAgent(FactualConfig(budget_limit_usd=5.0))
    """
    return FactualAgent(), SemanticAgent(), StructuralAgent()


def build_synthesis() -> SynthesisAgent:
    """Return a fresh synthesis (supervisor) agent with its default budget."""
    return SynthesisAgent()




# ════════════════════════════════════════════════════════════════════════
# MLFP06 — Exercise 6.2: Sequential Multi-Agent Pipeline (A → B → C)
# ════════════════════════════════════════════════════════════════════════
#
# WHAT YOU'LL LEARN:
#   - Build a 3-stage sequential agent pipeline where each stage feeds
#     the next (extract → interpret → synthesise)
#   - When sequential beats fan-out: downstream stages need upstream
#     structured output as input
#   - Latency model: sequential latency = SUM of stage latencies
#   - Contrast with supervisor-worker fan-out (independent specialists)
#
# PREREQUISITES: 01_supervisor_worker.py (you know the specialist agents)
# ESTIMATED TIME: ~25 min
#
# TASKS:
#   1. Load the corpus and wire a fresh specialist set
#   2. Instantiate the stage-2 InterpretationAgent
#   3. Build the 3-stage pipeline orchestrator
#   4. Run it and reason about sequential latency
#
# ════════════════════════════════════════════════════════════════════════


In [ ]:
from __future__ import annotations

import time

import matplotlib.pyplot as plt



## THEORY — Sequential Pipeline (A → B → C)


In [ ]:
# Sometimes specialists CANNOT run in parallel because each stage
# depends on the structured output of the previous one. Example:
#
#   Stage 1 (Factual extraction):   pull raw claims from the passage
#   Stage 2 (Interpretation):       interpret stage-1 claims in context
#   Stage 3 (Synthesis):            produce final answer from stage-2
#
# Stage 2 can't start until stage 1 has produced claims. Stage 3
# can't start until stage 2 has ranked them. This is a DAG with
# one linear edge per stage.
#
# Non-technical analogy: a factory assembly line for a handmade watch.
# The case maker has to finish before the movement fitter can begin,
# who has to finish before the dial painter can begin. You cannot
# parallelise the line without losing the dependency guarantees.
#
# LATENCY TRADE-OFF:
#   Supervisor-worker (concurrent fan-out, 01):
#       latency ≈ max(specialists) + synthesis
#       — when the backend serves requests in parallel; a single-slot
#         Ollama daemon queues them and you get close to the sum
#   Sequential pipeline:  latency ≈ sum(stages), on any backend
#
# You pay latency in exchange for the ability to CHAIN reasoning.



## TASK 1 — Load corpus and specialists


In [ ]:
print("=" * 70)
print("TASK 1: Load corpus + specialists")
print("=" * 70)

passages = load_squad_corpus()
factual_agent, _semantic_agent, _structural_agent = build_specialists()
synthesis_agent = build_synthesis()
print(f"Corpus: {passages.height} passages")
print("Specialists + synthesis supervisor instantiated")



### Checkpoint 1


In [ ]:
assert passages.height > 0, "Task 1: corpus should be non-empty"
assert factual_agent is not None, "Task 1: factual agent should exist"
print("✓ Checkpoint 1 passed\n")



## TASK 2 — Instantiate the stage-2 interpreter


In [ ]:
print("=" * 70)
print("TASK 2: Stage-2 InterpretationAgent")
print("=" * 70)

interpreter = InterpretationAgent()
print(f"  {interpreter.__class__.__name__}: {interpreter.description}")



### Checkpoint 2


In [ ]:
assert interpreter is not None, "Task 2: interpreter should exist"
print("✓ Checkpoint 2 passed\n")



## TASK 3 — Build the 3-stage sequential pipeline


Run 3-stage sequential pipeline: extract → interpret → synthesise.


In [ ]:
async def sequential_pipeline(doc: str, question: str) -> dict:
    t0 = time.perf_counter()
    per_stage: list[tuple[str, float]] = []

    # Stage 1: Factual extraction
    s1_t = time.perf_counter()
    factual = await run_checked(factual_agent, document=doc, question=question)
    per_stage.append(("factual extraction", time.perf_counter() - s1_t))

    # Stage 2: Interpretation (consumes stage-1 output)
    s2_t = time.perf_counter()
    interpreted = await run_checked(
        interpreter,
        factual_claims=str(factual["factual_claims"]),
        document=doc,
        question=question,
    )
    per_stage.append(("contextual interpretation", time.perf_counter() - s2_t))

    # Stage 3: Synthesis (consumes stage-2 output)
    s3_t = time.perf_counter()
    final = await run_checked(
        synthesis_agent,
        document=doc,
        question=question,
        factual_analysis=str(interpreted["interpreted_facts"]),
        semantic_analysis=str(interpreted["relevance_ranking"]),
        structural_analysis=f"Evidence quality: {factual['evidence_quality']}",
    )
    per_stage.append(("synthesis", time.perf_counter() - s3_t))

    elapsed = time.perf_counter() - t0
    return {
        "answer": final["unified_answer"],
        "confidence": final["confidence"],
        "stages": per_stage,
        "latency_s": elapsed,
    }



## TASK 4 — Run and inspect the latency breakdown


In [ ]:
doc = passages["text"][0]
question = passages["question"][0]
print(f"Question: {question}")

preflight_ollama(required_models=[MODEL])  # fails loudly if Ollama is down
result  = await sequential_pipeline(doc, question)

print(f"\nAnswer: {result['answer'][:250]}...")
print(f"Confidence: {result['confidence']:.2f}")
print(f"\nPer-stage latency:")
for name, dt in result["stages"]:
    print(f"  {name:30s} {dt:5.1f}s")
print(f"  {'TOTAL (sum of stages)':30s} {result['latency_s']:5.1f}s")

trace_path = OUTPUT_DIR / "ex6_sequential_pipeline_trace.txt"
trace_path.write_text(
    f"Question: {question}\n\nAnswer:\n{result['answer']}\n\n"
    + "\n".join(f"{n}: {t:.2f}s" for n, t in result["stages"])
    + f"\nTotal: {result['latency_s']:.2f}s\n"
)
print(f"\nTrace written to: {trace_path}")



### Checkpoint 3


In [ ]:
assert result["answer"], "Task 3: should produce a final answer"
assert len(result["stages"]) == 3, "Should log 3 stages"
print("\n✓ Checkpoint 3 passed — sequential pipeline complete\n")



## VISUALISE — Pipeline stage timing waterfall chart


In [ ]:
# Visual proof of the sequential latency model: total = sum of stages.
# The waterfall shows each stage starting where the previous one ended,
# making the dependency chain and its latency cost visually obvious.

stage_names = [name for name, _ in result["stages"]]
stage_times = [dt for _, dt in result["stages"]]
cumulative = [sum(stage_times[:i]) for i in range(len(stage_times))]

fig, ax = plt.subplots(figsize=(9, 4))
colors = ["#3498db", "#2ecc71", "#e67e22"]
for i, (start, dur, name) in enumerate(zip(cumulative, stage_times, stage_names)):
    ax.barh(
        0,
        dur,
        left=start,
        height=0.5,
        color=colors[i],
        edgecolor="white",
        label=f"{name} ({dur:.1f}s)",
    )
    ax.text(
        start + dur / 2,
        0,
        f"{dur:.1f}s",
        ha="center",
        va="center",
        fontsize=10,
        fontweight="bold",
        color="white",
    )

ax.set_yticks([])
ax.set_xlabel("Time (seconds)")
ax.set_title("Sequential Pipeline — Stage Timing Waterfall", fontweight="bold")
ax.legend(loc="upper right", fontsize=9)
ax.set_xlim(0, result["latency_s"] * 1.15)
plt.tight_layout()
fname = OUTPUT_DIR / "ex6_sequential_waterfall.png"
plt.savefig(fname, dpi=150, bbox_inches="tight")
plt.close(fig)
print(f"\n  Saved: {fname}")



## APPLY — Singapore scenario: regulatory filing analysis


Workload: 40 peer filings / quarter
  Baseline human review:      60 hours / quarter
  Sequential-pipeline assist: 2 hours / quarter (human approval only)
  Time saved:                 ~58 hours / quarter
  Fully-loaded analyst rate:  S$120/hour
  Quarterly savings:          ~S$7,000 per analyst
  Regulator benefit:          structured pipeline trace = SGX-defensible


In [ ]:
# SCENARIO (illustrative figures): A Singapore-listed REIT files quarterly disclosures with
# SGX. The compliance team reviews ~40 peer filings per quarter to
# benchmark disclosure quality. Today they read end-to-end — ~90
# minutes per filing, 60 hours per quarter.
#
# A sequential pipeline reframes this as:
#   Stage 1 (Factual):  extract numeric disclosures (NAV, DPU, gearing)
#   Stage 2 (Interpret): rank which disclosures matter for this quarter
#   Stage 3 (Synthesise): produce a benchmark brief with citations
#
# Why NOT supervisor-worker here: stage 2 needs stage-1's extracted
# numbers before it can rank relevance. Running "factual" and
# "interpret" in parallel would have interpret guessing at claims
# that haven't been extracted yet — the pipeline order is
# substantive, not decorative.
#
# IMPACT: reduces the 60-hour quarterly review to a ~2-hour human
# approval pass over pipeline drafts. At a fully-loaded compliance
# analyst rate of S$120/hour, that is ~S$7,000 saved per quarter per
# analyst, AND the audit trail is structured enough to present to
# SGX if a filing is queried.

print("=" * 70)
print("  SINGAPORE APPLICATION: REIT Peer Disclosure Benchmarking")
print("=" * 70)
print(
)



## DIAGNOSTIC CHECKPOINT


In [ ]:
# The diagnostic for a sequential pipeline is the stage waterfall you
# just plotted: total latency is the sum of the stages, so the widest
# bar is where optimisation pays.  A stage whose structured output is
# empty (no claims, no ranking) silently starves every later stage —
# check each stage's fields before blaming the synthesiser.



## REFLECTION


[x] Sequential multi-agent pipeline (A → B → C)
  [x] When to use sequential vs supervisor-worker fan-out
  [x] Latency model: sequential = sum of stages, parallel = max
  [x] Real downstream dependency: stage 2 consumes stage 1's structured
      output, stage 3 consumes stage 2's
  [x] REIT peer-filing benchmarking as a regulated Singapore use case

  KEY INSIGHT: Sequential is a choice, not a fallback. You pay
  latency to buy reasoning chain — each stage can interpret and
  REJECT upstream output, which fan-out specialists cannot do.

  Next: 03_parallel_router.py — running specialists truly in
  parallel with asyncio.gather, and letting an LLM route queries
  to the right specialist.


In [ ]:
print("=" * 70)
print("  WHAT YOU'VE MASTERED")
print("=" * 70)
print(
)

